In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
import re
import sys
import os

sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

# --- AS 4 LINHAS MÁGICAS QUE FALTAVAM ---
from src.neuralnet import NeuralNetwork
class NeuralNetworkTracked(NeuralNetwork):
    pass
sys.modules["__main__"].NeuralNetworkTracked = NeuralNetworkTracked
# ----------------------------------------

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

PATH_DATASET_TESTE = "../data/subm1.csv"
df_teste = pd.read_csv(PATH_DATASET_TESTE, sep=";")

with open("../modelo_numpy_artefactos.pkl", "rb") as f:
    le = pickle.load(f)["label_encoder"]

with open("../pytorch_vocab.pkl", "rb") as f:
    vocab = pickle.load(f)

class GRUClassifier(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_classes):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers=2,
                          batch_first=True, bidirectional=True, dropout=0.3)
        self.fc = nn.Linear(hidden_dim * 2, num_classes)
        self.dropout = nn.Dropout(0.4)
        
    def forward(self, x):
        embedded = self.embedding(x)
        _, hidden = self.gru(embedded)
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.fc(self.dropout(hidden))

model_pt = GRUClassifier(len(vocab), 128, 128, len(le.classes_)).to(device)
model_pt.load_state_dict(torch.load("../modelo_pytorch_gru.pth", map_location=device))
model_pt.eval()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"<.*?>", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    return text.split()

def encode_pt(text, vocab, max_len=100):
    tokens = clean_text(text)
    ids = [vocab.get(token, vocab["<unk>"]) for token in tokens][:max_len]
    if len(ids) < max_len:
        ids += [vocab["<pad>"]] * (max_len - len(ids))
    return ids

preds_pt_idx = []
with torch.no_grad():
    for text in df_teste["Text"].values:
        x = torch.tensor([encode_pt(text, vocab)], dtype=torch.long).to(device)
        output = model_pt(x)
        preds_pt_idx.append(torch.argmax(output, dim=1).item())

df_teste["Labels"] = le.inverse_transform(preds_pt_idx)

NOME_FICHEIRO_SAIDA = "subm1-g9-MEI-B.csv"
df_saida = df_teste[["ID", "Text", "Labels"]]
df_saida.to_csv(NOME_FICHEIRO_SAIDA, index=False, sep=";")

print(f"Previsões concluídas e guardadas em {NOME_FICHEIRO_SAIDA}")
df_saida.head()

Previsões concluídas e guardadas em subm1-g9-MEI-B.csv


,ID,Text,Labels
0,D2-1,A covalent bond is a chemical bond that involv...,OpenAI
1,D2-2,A covalent bond forms when two atoms share one...,OpenAI
2,D2-3,A covalent bond is a type of chemical bond whe...,OpenAI
3,D2-4,A covalent bond is a chemical bond that involv...,OpenAI
4,D2-5,Driven by exciting developments in the field o...,Human
